# TotalProject — COMP560 Face Recognition

Single notebook containing all project concepts. Run cells top-to-bottom.

**Recommended quick start:** Set `DEBUG = True` and `RUN_TRAIN = True` to smoke-test the full pipeline in ~10 min on a T4 GPU.

| Section | Contents |
|---|---|
| 0 | Colab setup (install, mount Drive, stage dataset) |
| 1 | **Master configuration** — edit this cell |
| 2 | Imports |
| 3 | Shared components (models, losses, datasets, utilities) |
| 3g | Template splits (disjoint train/test subsets) |
| 4 | ResNet baseline (no training) |
| 5 | Phase 1: Sub-center ArcFace training |
| 6 | Phase 2: Standard ArcFace training |
| 7 | Flexible training (ArcFace or Triplet + warmup) |
| 8 | Prediction generation |
| 9 | TAR@FAR evaluation + ROC plot |
| 10 | COMP560 grader evaluation |
| 11 | Ablation study (3×3 grid) |
| 12 | Save outputs to Google Drive |

## Section 0 — Colab Setup

Run these two cells once per session. Skip if `/content/data/dataset_a/` already exists.

In [ ]:
# Install dependencies and mount Google Drive
!pip install -q torch torchvision pandas pyarrow scikit-learn pillow tqdm matplotlib
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
mkdir -p /content/data/dataset_a
cp "/content/drive/MyDrive/COMP560/dataset_a/test.parquet"  /content/data/dataset_a/
cp "/content/drive/MyDrive/COMP560/dataset_a/pairs.parquet" /content/data/dataset_a/
tar -xzf "/content/drive/MyDrive/COMP560/dataset_a/images.tar.gz" -C /content/data/dataset_a/
echo "=== Staged dataset_a ==="
ls -lh /content/data/dataset_a

## Section 1 — Master Configuration

Edit the flags and hyperparameters below, then run all cells.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT   = "/content/data/dataset_a"            if IN_COLAB else "./datasets/dataset_a"
SAVE_DIR    = "/content/checkpoints"               if IN_COLAB else "./checkpoints"
RESULTS_DIR = "/content/results"                   if IN_COLAB else "./results"
PRED_OUTPUT = "/content/predictions/dataset_a.csv" if IN_COLAB else "predictions/dataset_a.csv"
CHECKPOINT  = f"{SAVE_DIR}/phase2.pth"             # used by predict + eval_tar

# ── Run mode — set True for the workflows you want ──────────────────────────
RUN_BASELINE = False  # ResNet50 pretrained baseline (no training)
RUN_TRAIN    = True   # Two-phase Sub-center ArcFace training
RUN_EXAMPLE  = False  # Flexible training (ArcFace or Triplet + warmup scheduler)
RUN_PREDICT  = True   # Generate predictions CSV from CHECKPOINT
RUN_EVAL_TAR = True   # TAR@FAR evaluation + optional ROC plot
RUN_EVALUATE = True   # COMP560 grader (score against ground truth pairs.parquet)
RUN_ABLATION = False  # 3x3 ablation grid (phase x embedding dim) — takes hours

# ── Training hyperparameters ─────────────────────────────────────────────────
PHASE         = "both"   # "1", "2", or "both"
SCHEDULE      = "step"   # "step" or "cosine"
EMBEDDING_DIM = 512      # 128, 256, or 512
EPOCHS_P1     = 8        # Phase 1 epochs
EPOCHS_P2     = 4        # Phase 2 epochs
BATCH_SIZE    = 128
NUM_WORKERS   = 0        # keep 0 to avoid multiprocessing errors in Jupyter
DEBUG         = False    # True: 500 images, 2 epochs — fast end-to-end smoke test

# ── Flexible training (RUN_EXAMPLE only) ────────────────────────────────────
LOSS          = "arcface"  # "arcface" or "triplet"
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
EPOCHS_EX     = 20
WARMUP_EPOCHS = 2
MARGIN        = 0.3        # triplet loss margin
SAVE_EVERY    = 5          # save periodic checkpoint every N epochs

# ── Evaluation ───────────────────────────────────────────────────────────────
CHECKPOINT_PATHS = [CHECKPOINT]  # eval_tar accepts a list — add multiple to compare
PLOT_ROC         = True
STUDENT_ID       = "YOUR_ID"
DATASETS         = ["dataset_a"]

# ── Subset configuration (revised branch) ──────────────────────────────────
SUBSET_SEED      = 42       # reproducibility
TRAIN_FRAC       = 0.15     # fraction of template_ids used for training
TEST_FRAC        = 0.10     # fraction of template_ids used for eval (disjoint)

import torch
if torch.cuda.is_available():           DEVICE = "cuda"
elif torch.backends.mps.is_available(): DEVICE = "mps"
else:                                   DEVICE = "cpu"
print(f"Device: {DEVICE}  |  IN_COLAB: {IN_COLAB}")

## Section 2 — Imports

In [ ]:
import itertools
import json
import math
import os
import random
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import roc_curve, auc
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from tqdm import tqdm

## Section 3 — Shared Components

All classes and functions are defined once here and reused by every section below.

### 3a — Models

In [ ]:
class FaceEncoder(nn.Module):
    """ResNet50 backbone with a linear projection head. Used for training."""
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.fc = nn.Linear(2048, embedding_dim)
        self._embedding_dim = embedding_dim

    @property
    def embedding_dim(self):
        return self._embedding_dim

    def forward(self, x):
        return self.backbone(x)

    def encode(self, x):
        return F.normalize(self.forward(x), p=2, dim=1)


class ResNetEncoder(nn.Module):
    """Pretrained ResNet50 with identity fc — no-training baseline."""
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.fc = nn.Identity()

    def encode(self, x):
        return F.normalize(self.backbone(x), p=2, dim=1)

### 3b — Augmentation & Transforms

In [ ]:
class GridMask:
    def __init__(self, grid=4, drop_ratio=0.25, p=0.15):
        self.grid = grid
        self.n_drop = math.ceil(drop_ratio * grid * grid)
        self.p = p

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor
        _, H, W = tensor.shape
        cell_h, cell_w = H // self.grid, W // self.grid
        cells = [(r, c) for r in range(self.grid) for c in range(self.grid)]
        tensor = tensor.clone()
        for r, c in random.sample(cells, self.n_drop):
            tensor[:, r*cell_h:(r+1)*cell_h, c*cell_w:(c+1)*cell_w] = 0.0
        return tensor


TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomResizedCrop(112, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
    GridMask(grid=4, drop_ratio=0.25, p=0.15),
])

EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

### 3c — Loss Functions

In [ ]:
class SubCenterArcFace(nn.Module):
    """Phase 1 loss: K sub-centers per identity, max cosine across sub-centers before margin."""
    def __init__(self, embedding_dim, num_classes, K=3, s=64.0, m=0.5):
        super().__init__()
        self.K, self.s, self.m, self.num_classes = K, s, m, num_classes
        self.weight = nn.Parameter(torch.FloatTensor(num_classes * K, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, emb, labels):
        B = emb.size(0)
        cosine = (F.linear(F.normalize(emb), F.normalize(self.weight))
                  .view(B, self.num_classes, self.K).max(dim=2).values)
        theta = torch.acos(torch.clamp(cosine, -1 + 1e-7, 1 - 1e-7))
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.unsqueeze(1), 1.0)
        return F.cross_entropy(torch.cos(theta + self.m * one_hot) * self.s, labels)

    def sub_center_cosines(self, emb):
        return (F.linear(F.normalize(emb), F.normalize(self.weight))
                .view(emb.size(0), self.num_classes, self.K))


class ArcFaceLoss(nn.Module):
    """Standard ArcFace margin loss. Used for Phase 2 and flexible training."""
    def __init__(self, embedding_dim, num_classes, s=64.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, emb, labels):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))
        theta = torch.acos(torch.clamp(cosine, -1 + 1e-7, 1 - 1e-7))
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.unsqueeze(1), 1.0)
        return F.cross_entropy(torch.cos(theta + self.m * one_hot) * self.s, labels)


class TripletLoss(nn.Module):
    """Hard-positive / hard-negative triplet loss. Used in flexible training."""
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, p=2, dim=1)
        dist_mat = 1 - torch.mm(embeddings, embeddings.t())
        same = labels.unsqueeze(0) == labels.unsqueeze(1)
        loss, count = torch.tensor(0.0, device=embeddings.device), 0
        for i in range(embeddings.size(0)):
            pos_mask = same[i].clone(); pos_mask[i] = False
            neg_mask = ~same[i]
            if pos_mask.any() and neg_mask.any():
                loss += F.relu(dist_mat[i][pos_mask].max() - dist_mat[i][neg_mask].min() + self.margin)
                count += 1
        return loss / max(count, 1)

### 3d — Data Loading

In [ ]:
def load_required_parquet(root, parquet_file, required_columns):
    path = os.path.join(root, parquet_file)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing parquet file: {path}")
    df = pd.read_parquet(path)
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{parquet_file} missing columns: {missing}")
    return df


def resolve_image_path(root, image_path):
    """Try three candidate locations for an image, returning the first that exists."""
    rel = str(image_path).replace("\\", "/").lstrip("./")
    for candidate in [
        os.path.join(root, rel),
        os.path.join(root, "images", rel),
        os.path.join(root, "images", os.path.basename(rel)),
    ]:
        if os.path.exists(candidate):
            return candidate
    return os.path.join(root, rel)


def split_templates(root, train_frac, test_frac, seed):
    """Return disjoint (train_tids, test_tids) sampled from test.parquet."""
    df = load_required_parquet(root, "test.parquet", ["template_id"])
    all_tids = sorted(df["template_id"].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(all_tids)
    n_train = max(1, int(len(all_tids) * train_frac))
    n_test  = max(1, int(len(all_tids) * test_frac))
    train_tids = set(all_tids[:n_train])
    test_tids  = set(all_tids[n_train : n_train + n_test])
    return train_tids, test_tids


class FaceTrainDataset(Dataset):
    def __init__(self, root, parquet_file="test.parquet", allowed_tids=None):
        self.root = root
        df = load_required_parquet(root, parquet_file, ["image_path", "template_id"])
        if allowed_tids is not None:
            df = df[df["template_id"].isin(allowed_tids)].reset_index(drop=True)
        unique_templates = sorted(df["template_id"].unique())
        self.tid_to_label = {tid: i for i, tid in enumerate(unique_templates)}
        self.num_classes = len(unique_templates)
        self.image_paths = df["image_path"].tolist()
        self.labels = [self.tid_to_label[tid] for tid in df["template_id"].values]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(resolve_image_path(self.root, self.image_paths[idx])).convert("RGB")
        return TRAIN_TRANSFORM(img), self.labels[idx]


class FaceTestDataset(Dataset):
    def __init__(self, root, allowed_tids=None):
        self.root = root
        df = load_required_parquet(root, "test.parquet", ["image_path", "template_id", "media_id"])
        if allowed_tids is not None:
            df = df[df["template_id"].isin(allowed_tids)].reset_index(drop=True)
        self.image_paths = df["image_path"].tolist()
        self.template_ids = df["template_id"].values
        self.media_ids = df["media_id"].values

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(resolve_image_path(self.root, self.image_paths[idx])).convert("RGB")
        return EVAL_TRANSFORM(img), idx

### 3e — Training Utilities

In [ ]:
def make_optimizer(model, criterion, lr=0.1, wd=5e-4):
    params = list(model.parameters()) + list(criterion.parameters())
    return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd)


def make_scheduler(optimizer, schedule, epochs, milestones=None):
    if schedule == "step":
        return torch.optim.lr_scheduler.MultiStepLR(
            optimizer, milestones=milestones or [20, 28, 32], gamma=0.1
        )
    return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )


def run_epoch(model, criterion, loader, optimizer, scheduler, device, desc):
    model.train()
    total = 0.0
    for imgs, labels in tqdm(loader, desc=desc, leave=True):
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(model(imgs), labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    scheduler.step()
    return total / len(loader)


def save_ckpt(path, model, epoch, loss, embedding_dim):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "loss": loss,
        "embedding_dim": embedding_dim,
    }, path)

### 3f — Evaluation Utilities

In [ ]:
def aggregate_template_features(embeddings, template_ids, media_ids):
    template_features = {}
    for tid in np.unique(template_ids):
        mask = template_ids == tid
        t_embs, t_mids = embeddings[mask], media_ids[mask]
        media_feats = [t_embs[t_mids == mid].mean(axis=0) for mid in np.unique(t_mids)]
        feat = np.sum(media_feats, axis=0)
        template_features[tid] = feat / (np.linalg.norm(feat) + 1e-8)
    return template_features


def encode_dataset(model, dataset, batch_size, num_workers, device):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=(device.type == "cuda"))
    embs, idxs = [], []
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    with torch.inference_mode():
        for imgs, indices in tqdm(loader, desc="Encoding"):
            embs.append(model.encode(imgs.to(device)).cpu().numpy())
            idxs.append(indices.numpy())
    elapsed = time.perf_counter() - t0
    embeddings = np.vstack(embs)[np.argsort(np.concatenate(idxs))]
    return embeddings, elapsed


def compute_scores(dataset, embeddings, pairs_df, embedding_dim):
    tf = aggregate_template_features(embeddings, dataset.template_ids, dataset.media_ids)
    tid_list = sorted(tf.keys())
    tid_to_idx = {tid: i for i, tid in enumerate(tid_list)}
    feat_matrix = np.zeros((len(tid_list), embedding_dim), dtype=np.float32)
    for tid, feat in tf.items():
        feat_matrix[tid_to_idx[tid]] = feat
    t1s = pairs_df["template_id_1"].values
    t2s = pairs_df["template_id_2"].values
    scores = np.zeros(len(pairs_df), dtype=np.float32)
    for i in range(0, len(pairs_df), 500_000):
        end = min(i + 500_000, len(pairs_df))
        scores[i:end] = np.sum(
            feat_matrix[np.array([tid_to_idx[t] for t in t1s[i:end]])] *
            feat_matrix[np.array([tid_to_idx[t] for t in t2s[i:end]])],
            axis=1,
        )
    return scores


def tar_at_far(pos_scores, neg_scores, far_targets=None):
    """Threshold-based TAR@FAR computation (no sklearn dependency)."""
    if far_targets is None:
        far_targets = [1e-4, 1e-5, 1e-6]
    neg_sorted = np.sort(neg_scores)[::-1]
    results = {}
    for far in far_targets:
        idx = max(math.ceil(far * len(neg_scores)) - 1, 0)
        results[f"TAR@FAR={far:.0e}"] = float((pos_scores > neg_sorted[idx]).mean()) * 100
    return results


def compute_tar_at_far_sklearn(scores, labels, far_targets=None):
    """sklearn roc_curve-based TAR@FAR + AUC. Used by the COMP560 grader."""
    if far_targets is None:
        far_targets = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
    fpr, tpr, _ = roc_curve(labels, scores)
    results = {f"TAR@FAR={far:.0e}": tpr[np.argmin(np.abs(fpr - far))] * 100
               for far in far_targets}
    results["AUC"] = auc(fpr, tpr) * 100
    return results

### 3g — Template Splits

In [ ]:
# Compute disjoint train/test template splits once; reused by all sections
_train_tids, _test_tids = split_templates(DATA_ROOT, TRAIN_FRAC, TEST_FRAC, SUBSET_SEED)
assert len(_train_tids & _test_tids) == 0, "Split overlap detected!"
print(f"Train templates: {len(_train_tids)}  |  Test templates: {len(_test_tids)}")

## Section 4 — ResNet Baseline

Pretrained ResNet50 with no fine-tuning. Saves predictions to `PRED_OUTPUT`. Runs only if `RUN_BASELINE = True`.

In [ ]:
if RUN_BASELINE:
    print("=== ResNet Baseline ===")
    device = torch.device(DEVICE)
    model_b = ResNetEncoder().to(device).eval()

    dataset_b  = FaceTestDataset(DATA_ROOT, allowed_tids=_test_tids)
    pairs_df_b = load_required_parquet(DATA_ROOT, "pairs.parquet", ["template_id_1", "template_id_2"])
    pairs_df_b = pairs_df_b[
        pairs_df_b["template_id_1"].isin(_test_tids) &
        pairs_df_b["template_id_2"].isin(_test_tids)
    ].reset_index(drop=True)
    print(f"Images: {len(dataset_b)}, Pairs: {len(pairs_df_b)}")

    embeddings_b, elapsed_b = encode_dataset(model_b, dataset_b, BATCH_SIZE, NUM_WORKERS, device)
    scores_b = compute_scores(dataset_b, embeddings_b, pairs_df_b, embeddings_b.shape[1])

    Path(PRED_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "template_id_1": pairs_df_b["template_id_1"].values,
        "template_id_2": pairs_df_b["template_id_2"].values,
        "score": scores_b,
    }).to_csv(PRED_OUTPUT, index=False)
    print(f"Baseline predictions saved: {PRED_OUTPUT}")
    print(f"Throughput: {len(dataset_b)/elapsed_b:.1f} img/s")

## Section 5 — Phase 1: Sub-center ArcFace Training

Trains with K=3 sub-centers per identity, then flags noisy samples via sub-center assignment consistency. Runs only if `RUN_TRAIN = True` and `PHASE` is `"1"` or `"both"`.

In [ ]:
_p1_ckpt_path = None
_flags_path   = None

if RUN_TRAIN and PHASE in ("1", "both"):
    print("=== Phase 1: Sub-center ArcFace ===")
    device = torch.device(DEVICE)
    Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

    full_ds    = FaceTrainDataset(DATA_ROOT, allowed_tids=_train_tids)
    num_classes = full_ds.num_classes
    epochs_p1  = 2 if DEBUG else EPOCHS_P1

    if DEBUG:
        dataset_p1 = Subset(full_ds, list(range(min(500, len(full_ds)))))
        dataset_p1.num_classes = num_classes
        print(f"DEBUG mode: 500 images, {epochs_p1} epochs")
    else:
        dataset_p1 = full_ds
    print(f"Dataset: {len(dataset_p1)} images, {num_classes} identities")

    model_p1     = FaceEncoder(EMBEDDING_DIM).to(device)
    criterion_p1 = SubCenterArcFace(EMBEDDING_DIM, num_classes).to(device)
    optimizer_p1 = make_optimizer(model_p1, criterion_p1)
    scheduler_p1 = make_scheduler(optimizer_p1, SCHEDULE, epochs_p1, milestones=[20, 28, 32])
    loader_p1    = DataLoader(dataset_p1, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"), drop_last=True)

    for ep in range(epochs_p1):
        avg = run_epoch(model_p1, criterion_p1, loader_p1, optimizer_p1, scheduler_p1,
                        device, f"Ph1 {ep+1}/{epochs_p1}")
        print(f"  epoch {ep+1}  loss={avg:.4f}")

    _p1_ckpt_path = str(Path(SAVE_DIR) / "phase1.pth")
    save_ckpt(_p1_ckpt_path, model_p1, epochs_p1 - 1, avg, EMBEDDING_DIM)
    print(f"Saved: {_p1_ckpt_path}")

    # Noise isolation via sub-center assignment consistency
    print("Running noise isolation...")
    model_p1.eval()
    sub_loader = DataLoader(dataset_p1, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    assignments = []
    with torch.inference_mode():
        for imgs, labels in tqdm(sub_loader, desc="Sub-center assignment"):
            cos = criterion_p1.sub_center_cosines(model_p1(imgs.to(device)))
            for i, lab in enumerate(labels.tolist()):
                assignments.append((lab, cos[i, lab].argmax().item()))

    identity_sub = {}
    for lab, k in assignments:
        identity_sub.setdefault(lab, []).append(k)
    dominant = {lab: Counter(ks).most_common(1)[0][0] for lab, ks in identity_sub.items()}
    noise_flags = np.array([assignments[i][1] != dominant[assignments[i][0]]
                            for i in range(len(assignments))])

    _flags_path = str(Path(SAVE_DIR) / "noise_flags.npy")
    np.save(_flags_path, noise_flags)
    n_noise = int(noise_flags.sum())
    print(f"Flagged {n_noise}/{len(noise_flags)} samples as noise ({100*n_noise/len(noise_flags):.1f}%)")

## Section 6 — Phase 2: Standard ArcFace Training

Loads noise flags from Phase 1, trains on the clean subset with standard ArcFace. Optionally initializes from `phase1.pth`. Runs only if `RUN_TRAIN = True` and `PHASE` is `"2"` or `"both"`.

In [ ]:
if RUN_TRAIN and PHASE in ("2", "both"):
    print("=== Phase 2: Standard ArcFace ===")
    device = torch.device(DEVICE)
    Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

    full_ds_p2   = FaceTrainDataset(DATA_ROOT, allowed_tids=_train_tids)
    num_classes_p2 = full_ds_p2.num_classes
    epochs_p2    = 2 if DEBUG else EPOCHS_P2

    flags_file = _flags_path or str(Path(SAVE_DIR) / "noise_flags.npy")
    if os.path.exists(flags_file):
        noise_flags_p2 = np.load(flags_file)
        clean_idx = np.where(~noise_flags_p2)[0].tolist()
        if DEBUG:
            clean_idx = [i for i in clean_idx if i < 500]
        dataset_p2 = Subset(full_ds_p2, clean_idx)
        dataset_p2.num_classes = num_classes_p2
        print(f"Phase 2: {len(dataset_p2)} clean samples (dropped {int(noise_flags_p2.sum())} noisy)")
    else:
        dataset_p2 = Subset(full_ds_p2, list(range(min(500, len(full_ds_p2))))) if DEBUG else full_ds_p2
        if DEBUG: dataset_p2.num_classes = num_classes_p2
        print("No noise_flags.npy found — using full dataset")

    model_p2 = FaceEncoder(EMBEDDING_DIM).to(device)
    init_ckpt = _p1_ckpt_path or str(Path(SAVE_DIR) / "phase1.pth")
    if os.path.exists(init_ckpt):
        ckpt_data = torch.load(init_ckpt, map_location=device, weights_only=True)
        model_p2.load_state_dict(ckpt_data["model_state_dict"])
        print(f"Initialized from {init_ckpt}")

    criterion_p2 = ArcFaceLoss(EMBEDDING_DIM, num_classes_p2).to(device)
    optimizer_p2 = make_optimizer(model_p2, criterion_p2)
    scheduler_p2 = make_scheduler(optimizer_p2, SCHEDULE, epochs_p2, milestones=[8, 12, 14])
    loader_p2    = DataLoader(dataset_p2, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"), drop_last=True)

    for ep in range(epochs_p2):
        avg = run_epoch(model_p2, criterion_p2, loader_p2, optimizer_p2, scheduler_p2,
                        device, f"Ph2 {ep+1}/{epochs_p2}")
        print(f"  epoch {ep+1}  loss={avg:.4f}")

    _p2_ckpt_path = str(Path(SAVE_DIR) / "phase2.pth")
    save_ckpt(_p2_ckpt_path, model_p2, epochs_p2 - 1, avg, EMBEDDING_DIM)
    print(f"Saved: {_p2_ckpt_path}")

## Section 7 — Flexible Training (ArcFace or Triplet + Warmup)

Alternative training with AdamW optimizer, linear warmup + cosine annealing. Set `LOSS = "arcface"` or `"triplet"`. Saves `best_model.pth`. Runs only if `RUN_EXAMPLE = True`.

In [ ]:
if RUN_EXAMPLE:
    print(f"=== Flexible Training ({LOSS}) ===")
    device = torch.device(DEVICE)
    Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

    epochs_ex = 2 if DEBUG else EPOCHS_EX
    base_ds_ex = FaceTrainDataset(DATA_ROOT, allowed_tids=_train_tids)
    if DEBUG:
        dataset_ex = Subset(base_ds_ex, list(range(min(500, len(base_ds_ex)))))
        dataset_ex.num_classes = base_ds_ex.num_classes
    else:
        dataset_ex = base_ds_ex
    loader_ex = DataLoader(dataset_ex, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"), drop_last=True)
    print(f"Dataset: {len(dataset_ex)} images, {dataset_ex.num_classes} identities")

    model_ex = FaceEncoder(EMBEDDING_DIM).to(device)
    if LOSS == "arcface":
        criterion_ex = ArcFaceLoss(EMBEDDING_DIM, dataset_ex.num_classes, s=30.0, m=0.5).to(device)
    elif LOSS == "triplet":
        criterion_ex = TripletLoss(margin=MARGIN)
    else:
        raise ValueError(f"Unknown LOSS: {LOSS!r}. Use 'arcface' or 'triplet'.")

    params_ex = list(model_ex.parameters())
    if hasattr(criterion_ex, "parameters"):
        params_ex += list(criterion_ex.parameters())
    optimizer_ex  = torch.optim.AdamW(params_ex, lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps   = len(loader_ex) * epochs_ex
    warmup_steps  = len(loader_ex) * WARMUP_EPOCHS
    lr_lambda     = lambda s: (s / max(warmup_steps, 1)) if s < warmup_steps else \
                              0.5 * (1 + math.cos(math.pi * (s - warmup_steps) / max(total_steps - warmup_steps, 1)))
    scheduler_ex  = torch.optim.lr_scheduler.LambdaLR(optimizer_ex, lr_lambda)

    best_loss_ex = float("inf")
    for ep in range(epochs_ex):
        model_ex.train()
        epoch_loss = 0.0
        for imgs, labels in tqdm(loader_ex, desc=f"Epoch {ep+1}/{epochs_ex}"):
            imgs, labels = imgs.to(device), labels.to(device)
            loss = criterion_ex(model_ex(imgs), labels)
            optimizer_ex.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model_ex.parameters(), 1.0)
            optimizer_ex.step(); scheduler_ex.step()
            epoch_loss += loss.item()
        avg_ex = epoch_loss / len(loader_ex)
        print(f"  epoch {ep+1}  loss={avg_ex:.4f}")
        if avg_ex < best_loss_ex:
            best_loss_ex = avg_ex
            save_ckpt(str(Path(SAVE_DIR) / "best_model.pth"), model_ex, ep, avg_ex, EMBEDDING_DIM)
            print(f"    -> best model saved (loss={avg_ex:.4f})")
        if (ep + 1) % SAVE_EVERY == 0:
            save_ckpt(str(Path(SAVE_DIR) / f"checkpoint_epoch{ep+1}.pth"), model_ex, ep, avg_ex, EMBEDDING_DIM)
    print(f"Flexible training done. Best loss: {best_loss_ex:.4f}")

## Section 8 — Prediction Generation

Loads a trained checkpoint, encodes all test images, aggregates template features, scores all pairs, and saves a predictions CSV. Runs only if `RUN_PREDICT = True`.

In [ ]:
if RUN_PREDICT:
    print("=== Prediction Generation ===")
    device = torch.device(DEVICE)

    ckpt_pred  = torch.load(CHECKPOINT, map_location=device, weights_only=True)
    dim_pred   = ckpt_pred.get("embedding_dim", EMBEDDING_DIM)
    model_pred = FaceEncoder(dim_pred).to(device)
    model_pred.load_state_dict(ckpt_pred["model_state_dict"])
    model_pred.eval()
    print(f"Loaded: {CHECKPOINT}  (dim={dim_pred})")

    dataset_pred  = FaceTestDataset(DATA_ROOT, allowed_tids=_test_tids)
    pairs_df_pred = load_required_parquet(DATA_ROOT, "pairs.parquet", ["template_id_1", "template_id_2"])
    pairs_df_pred = pairs_df_pred[
        pairs_df_pred["template_id_1"].isin(_test_tids) &
        pairs_df_pred["template_id_2"].isin(_test_tids)
    ].reset_index(drop=True)
    print(f"Images: {len(dataset_pred)}, Pairs: {len(pairs_df_pred)}")

    embeddings_pred, elapsed_pred = encode_dataset(model_pred, dataset_pred, BATCH_SIZE, NUM_WORKERS, device)
    print(f"Encoded in {elapsed_pred:.1f}s  ({len(dataset_pred)/elapsed_pred:.0f} img/s)")

    scores_pred = compute_scores(dataset_pred, embeddings_pred, pairs_df_pred, dim_pred)

    Path(PRED_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "template_id_1": pairs_df_pred["template_id_1"].values,
        "template_id_2": pairs_df_pred["template_id_2"].values,
        "score": scores_pred,
    }).to_csv(PRED_OUTPUT, index=False)
    print(f"Predictions saved: {PRED_OUTPUT}  ({len(scores_pred)} pairs)")

## Section 9 — TAR@FAR Evaluation + ROC Plot

Evaluates one or more checkpoints. Reports TAR@FAR=1e-4/1e-5/1e-6, throughput, peak GPU memory. Saves ROC plot if `PLOT_ROC = True`. Runs only if `RUN_EVAL_TAR = True`.

In [ ]:
if RUN_EVAL_TAR:
    print("=== TAR@FAR Evaluation ===")
    device = torch.device(DEVICE)
    Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

    rows_tar, roc_data = [], []
    pairs_df_tar = load_required_parquet(DATA_ROOT, "pairs.parquet",
                                         ["template_id_1", "template_id_2", "label"])
    pairs_df_tar = pairs_df_tar[
        pairs_df_tar["template_id_1"].isin(_test_tids) &
        pairs_df_tar["template_id_2"].isin(_test_tids)
    ].reset_index(drop=True)
    print(f"Eval pairs (within test split): {len(pairs_df_tar)}")

    for ckpt_path in CHECKPOINT_PATHS:
        print(f"\n--- {ckpt_path} ---")
        ckpt_tar  = torch.load(ckpt_path, map_location=device, weights_only=True)
        dim_tar   = ckpt_tar.get("embedding_dim", EMBEDDING_DIM)
        model_tar = FaceEncoder(dim_tar).to(device)
        model_tar.load_state_dict(ckpt_tar["model_state_dict"])
        model_tar.eval()

        dataset_tar = FaceTestDataset(DATA_ROOT, allowed_tids=_test_tids)
        embeddings_tar, elapsed_tar = encode_dataset(model_tar, dataset_tar, BATCH_SIZE, NUM_WORKERS, device)
        throughput  = len(dataset_tar) / elapsed_tar
        peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if device.type == "cuda" else 0.0

        scores_tar = compute_scores(dataset_tar, embeddings_tar, pairs_df_tar, dim_tar)
        labels_tar = pairs_df_tar["label"].values
        pos_tar    = scores_tar[labels_tar == 1]
        neg_tar    = scores_tar[labels_tar == 0]

        metrics_tar = tar_at_far(pos_tar, neg_tar)
        metrics_tar.update({"throughput_img_s": throughput, "peak_gpu_mem_mb": peak_mem_mb,
                             "embedding_dim": dim_tar, "eval_time_s": elapsed_tar,
                             "checkpoint": ckpt_path})
        rows_tar.append(metrics_tar)
        roc_data.append((ckpt_path, scores_tar, labels_tar))

        for k in ["TAR@FAR=1e-04", "TAR@FAR=1e-05", "TAR@FAR=1e-06"]:
            print(f"  {k}: {metrics_tar.get(k, 0):.2f}%")
        print(f"  Throughput: {throughput:.1f} img/s  |  Peak GPU: {peak_mem_mb:.1f} MB")

    if PLOT_ROC and roc_data:
        fig, ax = plt.subplots(figsize=(8, 6))
        for ckpt_path, scores, labels in roc_data:
            pos  = scores[labels == 1]; neg = scores[labels == 0]
            thresholds = np.sort(np.unique(scores))[::-1]
            fprs = [(neg >= tau).mean() for tau in thresholds]
            tprs = [(pos >= tau).mean() for tau in thresholds]
            ax.plot(fprs, tprs, label=Path(ckpt_path).stem)
        ax.set_xscale("log"); ax.set_xlabel("FAR (log scale)"); ax.set_ylabel("TAR")
        ax.set_title("ROC Curve"); ax.legend(); ax.grid(True, which="both", alpha=0.3)
        roc_path = str(Path(RESULTS_DIR) / "roc.png")
        fig.savefig(roc_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"ROC saved: {roc_path}")

## Section 10 — COMP560 Grader Evaluation

Scores your predictions CSV against ground truth `pairs.parquet`. Generates a JSON report and summary CSV. Runs only if `RUN_EVALUATE = True`.

In [ ]:
if RUN_EVALUATE:
    print("=== COMP560 Grader ===")
    Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    all_results = {"student_id": STUDENT_ID, "timestamp": timestamp,
                   "prediction_path": PRED_OUTPUT, "datasets": {}}

    for dataset_name in DATASETS:
        print(f"\n--- {dataset_name} ---")
        if dataset_name == "dataset_a":
            gt_root = DATA_ROOT
        else:
            gt_root = DATA_ROOT.replace("dataset_a", dataset_name)
        gt_path = os.path.join(gt_root, "pairs.parquet")

        if not os.path.exists(gt_path):
            print(f"  Ground truth not found: {gt_path}")
            all_results["datasets"][dataset_name] = {"error": "ground truth not found"}
            continue

        pred_path = os.path.join(PRED_OUTPUT, f"{dataset_name}.csv") \
                    if os.path.isdir(PRED_OUTPUT) else PRED_OUTPUT
        if not os.path.exists(pred_path):
            print(f"  Prediction file not found: {pred_path}")
            all_results["datasets"][dataset_name] = {"error": "prediction file not found"}
            continue

        try:
            gt_df   = pd.read_parquet(gt_path)
            gt_df   = gt_df[
                gt_df["template_id_1"].isin(_test_tids) &
                gt_df["template_id_2"].isin(_test_tids)
            ].reset_index(drop=True)
            pred_df = pd.read_csv(pred_path)
            missing_cols = {"template_id_1", "template_id_2", "score"} - set(pred_df.columns)
            if missing_cols:
                raise ValueError(f"Prediction CSV missing columns: {missing_cols}")

            gt_df["pair_key"]   = gt_df["template_id_1"].astype(str) + "_" + gt_df["template_id_2"].astype(str)
            pred_df["pair_key"] = pred_df["template_id_1"].astype(str) + "_" + pred_df["template_id_2"].astype(str)
            merged = gt_df.merge(pred_df[["pair_key", "score"]], on="pair_key", how="left")
            n_missing = int(merged["score"].isna().sum())
            if n_missing:
                print(f"  WARNING: {n_missing} missing predictions (score=0 used)")
            merged["score"] = merged["score"].fillna(0.0)

            perf = compute_tar_at_far_sklearn(merged["score"].values, merged["label"].values)
            all_results["datasets"][dataset_name] = {
                "performance": perf,
                "submission_info": {
                    "num_predicted_pairs": len(pred_df),
                    "num_gt_pairs": len(gt_df),
                    "num_missing_pairs": n_missing,
                    "num_positive_pairs": int((merged["label"] == 1).sum()),
                    "num_negative_pairs": int((merged["label"] == 0).sum()),
                },
            }
            print("Performance:")
            for metric, val in perf.items():
                print(f"  {metric}: {val:.2f}%")
        except Exception as e:
            import traceback; traceback.print_exc()
            all_results["datasets"][dataset_name] = {"error": str(e)}

    result_file = Path(RESULTS_DIR) / f"{STUDENT_ID}_{timestamp}.json"
    with open(result_file, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nResults saved: {result_file}")

## Section 11 — Ablation Study

Runs a 3×3 grid: `{phase 1, phase 2, both} × {D=128, 256, 512}`. Reuses all shared components. Saves summary CSV and displays results. Set `DEBUG = True` for a fast smoke test.

**Warning:** Full run takes several hours (`DEBUG = False`). Runs only if `RUN_ABLATION = True`.

In [ ]:
if RUN_ABLATION:
    print("=== Ablation Study ===")
    device      = torch.device(DEVICE)
    ablation_dir = str(Path(RESULTS_DIR) / "ablation")
    Path(ablation_dir).mkdir(parents=True, exist_ok=True)

    phases   = ["1", "2", "both"]
    dims     = [128, 256, 512]
    summary_rows = []

    pairs_df_ab = load_required_parquet(DATA_ROOT, "pairs.parquet",
                                        ["template_id_1", "template_id_2", "label"])
    pairs_df_ab = pairs_df_ab[
        pairs_df_ab["template_id_1"].isin(_test_tids) &
        pairs_df_ab["template_id_2"].isin(_test_tids)
    ].reset_index(drop=True)
    full_ds_ab  = FaceTrainDataset(DATA_ROOT, allowed_tids=_train_tids)
    num_cls_ab  = full_ds_ab.num_classes

    for phase_ab, dim_ab in itertools.product(phases, dims):
        tag      = f"phase{phase_ab}_d{dim_ab}"
        tag_save = str(Path(SAVE_DIR) / "ablation" / tag)
        Path(tag_save).mkdir(parents=True, exist_ok=True)
        print(f"\n{'='*50}\nphase={phase_ab}  dim={dim_ab}")
        ep_ab = 2 if DEBUG else EPOCHS_P1
        ep2_ab = 2 if DEBUG else EPOCHS_P2

        try:
            # Dataset slice for debug
            if DEBUG:
                ds_ab = Subset(full_ds_ab, list(range(min(500, len(full_ds_ab)))))
                ds_ab.num_classes = num_cls_ab
            else:
                ds_ab = full_ds_ab
            lkw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                       pin_memory=(device.type == "cuda"), drop_last=True)

            ab_p1_ckpt = None; ab_flags = None; ab_final_ckpt = None

            # Phase 1
            if phase_ab in ("1", "both"):
                m = FaceEncoder(dim_ab).to(device)
                c = SubCenterArcFace(dim_ab, num_cls_ab).to(device)
                opt = make_optimizer(m, c)
                sch = make_scheduler(opt, SCHEDULE, ep_ab, milestones=[20, 28, 32])
                for ep in range(ep_ab):
                    avg_ab = run_epoch(m, c, DataLoader(ds_ab, shuffle=True, **lkw),
                                       opt, sch, device, f"[Ab ph1 d={dim_ab}] {ep+1}/{ep_ab}")
                ab_p1_ckpt = str(Path(tag_save) / "phase1.pth")
                save_ckpt(ab_p1_ckpt, m, ep_ab - 1, avg_ab, dim_ab)
                ab_final_ckpt = ab_p1_ckpt

                # Noise isolation
                m.eval(); asgn = []
                with torch.inference_mode():
                    for imgs, labs in DataLoader(ds_ab, batch_size=BATCH_SIZE,
                                                 shuffle=False, num_workers=NUM_WORKERS):
                        cos = c.sub_center_cosines(m(imgs.to(device)))
                        for i, lab in enumerate(labs.tolist()):
                            asgn.append((lab, cos[i, lab].argmax().item()))
                id_sub = {}
                for lab, k in asgn: id_sub.setdefault(lab, []).append(k)
                dom = {lab: Counter(ks).most_common(1)[0][0] for lab, ks in id_sub.items()}
                ab_flags = np.array([asgn[i][1] != dom[asgn[i][0]] for i in range(len(asgn))])
                np.save(str(Path(tag_save) / "noise_flags.npy"), ab_flags)

            # Phase 2
            if phase_ab in ("2", "both"):
                if ab_flags is not None:
                    clean_idx_ab = np.where(~ab_flags)[0].tolist()
                    if DEBUG: clean_idx_ab = [i for i in clean_idx_ab if i < 500]
                    ds2_ab = Subset(full_ds_ab, clean_idx_ab); ds2_ab.num_classes = num_cls_ab
                else:
                    ds2_ab = ds_ab
                m2 = FaceEncoder(dim_ab).to(device)
                if ab_p1_ckpt and os.path.exists(ab_p1_ckpt):
                    ck = torch.load(ab_p1_ckpt, map_location=device, weights_only=True)
                    m2.load_state_dict(ck["model_state_dict"])
                c2 = ArcFaceLoss(dim_ab, num_cls_ab).to(device)
                opt2 = make_optimizer(m2, c2)
                sch2 = make_scheduler(opt2, SCHEDULE, ep2_ab, milestones=[8, 12, 14])
                for ep in range(ep2_ab):
                    avg_ab = run_epoch(m2, c2, DataLoader(ds2_ab, shuffle=True, **lkw),
                                       opt2, sch2, device, f"[Ab ph2 d={dim_ab}] {ep+1}/{ep2_ab}")
                ab_final_ckpt = str(Path(tag_save) / "phase2.pth")
                save_ckpt(ab_final_ckpt, m2, ep2_ab - 1, avg_ab, dim_ab)

            # Evaluation
            ck_ab  = torch.load(ab_final_ckpt, map_location=device, weights_only=True)
            d_ab   = ck_ab.get("embedding_dim", dim_ab)
            m_eval = FaceEncoder(d_ab).to(device); m_eval.load_state_dict(ck_ab["model_state_dict"]); m_eval.eval()
            ds_eval_ab = FaceTestDataset(DATA_ROOT, allowed_tids=_test_tids)
            embs_ab, el_ab = encode_dataset(m_eval, ds_eval_ab, BATCH_SIZE, NUM_WORKERS, device)
            sc_ab  = compute_scores(ds_eval_ab, embs_ab, pairs_df_ab, d_ab)
            labs_ab = pairs_df_ab["label"].values
            met_ab  = tar_at_far(sc_ab[labs_ab == 1], sc_ab[labs_ab == 0])
            met_ab.update({"phase": phase_ab, "dim": dim_ab,
                           "throughput_img_s": len(ds_eval_ab) / el_ab})
            summary_rows.append(met_ab)
            print(f"  TAR@FAR=1e-4: {met_ab.get('TAR@FAR=1e-04', 0):.2f}%  "
                  f"TAR@FAR=1e-5: {met_ab.get('TAR@FAR=1e-05', 0):.2f}%")
        except Exception as e:
            import traceback; traceback.print_exc()
            summary_rows.append({"phase": phase_ab, "dim": dim_ab, "error": str(e)})

    if summary_rows:
        summary_df = pd.DataFrame(summary_rows)
        cols = [c for c in ["phase", "dim", "TAR@FAR=1e-04", "TAR@FAR=1e-05", "TAR@FAR=1e-06",
                             "throughput_img_s"] if c in summary_df.columns]
        summary_path = str(Path(ablation_dir) / "summary.csv")
        summary_df[cols].to_csv(summary_path, index=False)
        print(f"\nAblation summary saved: {summary_path}")
        display(summary_df[cols])

## Section 12 — Save Outputs to Google Drive

Copies all checkpoints, predictions, and results back to Drive so they persist after the session ends. Safe to run multiple times.

In [ ]:
if IN_COLAB:
    import shutil
    for src, dst in [
        ("/content/checkpoints", "/content/drive/MyDrive/COMP560/checkpoints"),
        ("/content/predictions",  "/content/drive/MyDrive/COMP560/predictions"),
        ("/content/results",      "/content/drive/MyDrive/COMP560/results"),
    ]:
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"Saved {src} -> {dst}")
    print("All outputs saved to Drive.")
else:
    print("Not in Colab — outputs already in local directories.")